# CyberVision Gen — Personne 3 : entraînement des modèles A et B (Colab)

Ce notebook **lance** le script `cybervision_plausibility/entrainement.py`, testé en local.
Il ne contient pas de code d'IA : tout est dans le script.

- **Modèle A** : λ = 0 → identique au modèle de la Personne 2 (prouvé par `test_entrainement.py`).
- **Modèle B** : λ > 0 → même modèle + pénalité de plausibilité.

**Avant de commencer** (une seule fois) :
1. Menu *Exécution → Modifier le type d'exécution → GPU (T4)*.
2. Sur le Drive : raccourci du dossier partagé `CyberVision_Data` dans *Mon Drive*.
3. Sur le Drive : dossier `CyberVision_P3` contenant `cybervision-gen.zip`
   (créé en local avec `python cybervision_plausibility/creer_zip_colab.py`).

Exécuter les cellules **dans l'ordre**. Le test n'est jamais chargé.

## 1. Vérifier le GPU

In [ ]:
import torch
print("PyTorch", torch.__version__, "| GPU disponible :", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activez le GPU : Exécution > Modifier le type d'exécution > GPU"
print(torch.cuda.get_device_name(0))

## 2. Monter le Drive et vérifier les fichiers

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
ZIP = DRIVE / "CyberVision_P3" / "cybervision-gen.zip"
DONNEES_DRIVE = DRIVE / "CyberVision_Data"
SORTIES = DRIVE / "CyberVision_P3" / "sorties"       # checkpoints conservés sur le Drive

assert ZIP.exists(), f"Zip introuvable : {ZIP}"
for nom in ("train_scaled.parquet", "val_scaled.parquet"):
    assert (DONNEES_DRIVE / nom).exists(), f"{nom} introuvable dans {DONNEES_DRIVE} (raccourci créé ?)"
SORTIES.mkdir(parents=True, exist_ok=True)
print("OK — zip, données et dossier de sortie trouvés")

## 3. Installer le code et copier les données sur Colab

In [ ]:
import shutil, zipfile, os
PROJET = Path("/content/cybervision-gen")
shutil.rmtree(PROJET, ignore_errors=True)            # repartir d'une copie propre du code
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall("/content")

DONNEES = PROJET / "cybervision_data" / "data" / "processed"
DONNEES.mkdir(parents=True, exist_ok=True)
for nom in ("train_scaled.parquet", "val_scaled.parquet"):
    shutil.copy2(DONNEES_DRIVE / nom, DONNEES / nom)  # lecture locale = plus rapide

os.chdir(PROJET)
print("Code dans", PROJET, "| données :", sorted(p.name for p in DONNEES.glob("*.parquet")))

## 4. Lancer les tests (attendu : `11 passed`)

In [ ]:
!pip -q install pytest
!python -m pytest -q -p no:warnings cybervision_plausibility

## 5. Test rapide sur GPU (5 batches)
Vérifie que tout fonctionne sur Colab avant les longs entraînements.

In [ ]:
!python cybervision_plausibility/entrainement.py --modele A --smoke --sortie /content/smoke_A

## 6. Entraînement complet du modèle A (100 époques)
Chaque époque affiche une ligne de résultats (dont `duree_epoque_s`).
**En cas de déconnexion** : réexécuter les cellules 1 à 3, puis celle-ci — `--reprendre`
repart du dernier checkpoint enregistré sur le Drive.

In [ ]:
!python cybervision_plausibility/entrainement.py --modele A --reprendre --sortie "{SORTIES}/modele_A"

## 6 b. Résumé du modèle A
Meilleure époque (critère de la Personne 2 : distance de validation la plus faible).

In [ ]:
import json, pandas as pd

def lire_metrics(dossier):
    return pd.DataFrame(json.loads((Path(dossier) / "metrics.json").read_text(encoding="utf-8")))

hist_A = lire_metrics(SORTIES / "modele_A")
meilleure = hist_A.loc[hist_A["val_distribution_distance"].idxmin()]
print(f"Époques entraînées : {len(hist_A)}")
print(f"Meilleure époque   : {int(meilleure['epoch'])} | distance validation = {meilleure['val_distribution_distance']:.4f}"
      f" | pénalité validation = {meilleure['val_penalite_plausibilite']:.4f}")

## 7. Essais de λ (10 époques chacun)
Trois essais courts : λ = 0,1 / 1 / 10 (environ 7 minutes chacun).
En cas de déconnexion : réexécuter les cellules 1 à 3, puis celle-ci (`--reprendre`).

In [ ]:
LAMBDAS = [0.1, 1, 10]
for lam in LAMBDAS:
    print(f"===== Essai λ = {lam} =====")
    !python cybervision_plausibility/entrainement.py --modele B --lambda {lam} --epoques 10 --reprendre --sortie "{SORTIES}/essais/lambda_{lam}"

## 8. Choix de λ (règle fixée AVANT de voir les résultats)
On compare chaque essai au modèle A **aux mêmes époques** (moyenne des époques 8 à 10, pour lisser les variations) :
- ressemblance : `val_distribution_distance` (plus petit = mieux) ;
- plausibilité : `val_penalite_plausibilite` (plus petit = mieux).

**Règle** : parmi les λ dont la distance n'est pas plus de 10 % supérieure à celle de A,
on choisit celui qui a la plus petite pénalité. Uniquement la validation (jamais le test).

In [ ]:
TOLERANCE = 0.10                     # distance au plus 10 % moins bonne que A
EPOQUES = [7, 8, 9]                  # époques 8 à 10 (numérotées à partir de 0)

def moyenne(hist, colonne):
    return hist.loc[hist["epoch"].isin(EPOQUES), colonne].mean()

lignes = [{"lambda": 0.0, "modele": "A",
           "distance": moyenne(hist_A, "val_distribution_distance"),
           "penalite": moyenne(hist_A, "val_penalite_plausibilite")}]
for lam in LAMBDAS:
    h = lire_metrics(SORTIES / "essais" / f"lambda_{lam}")
    assert len(h) >= 10, f"Essai λ={lam} incomplet ({len(h)} époques) : relancer la cellule 7"
    lignes.append({"lambda": lam, "modele": "B",
                   "distance": moyenne(h, "val_distribution_distance"),
                   "penalite": moyenne(h, "val_penalite_plausibilite")})
tableau = pd.DataFrame(lignes)
dist_A = tableau.loc[0, "distance"]
tableau["distance_vs_A"] = (tableau["distance"] / dist_A - 1).map(lambda v: f"{100 * v:+.1f} %")
tableau["admissible"] = tableau["distance"] <= (1 + TOLERANCE) * dist_A
print(tableau.to_string(index=False))

candidats = tableau[(tableau["modele"] == "B") & tableau["admissible"]]
if len(candidats):
    LAMBDA_CHOISI = float(candidats.loc[candidats["penalite"].idxmin(), "lambda"])
    print(f"\nλ choisi : {LAMBDA_CHOISI}")
else:
    LAMBDA_CHOISI = None
    print("\nAucun λ admissible : tous dégradent la ressemblance de plus de 10 %. À discuter avant de lancer B.")

## 9. Entraînement complet du modèle B (100 époques)
Utilise le λ choisi à la cellule 8. **Vérifier le tableau ci-dessus avant de lancer.**
En cas de déconnexion : réexécuter les cellules 1 à 3, 6 b, 8, puis celle-ci (`--reprendre`).

In [ ]:
LAMBDA = LAMBDA_CHOISI
assert LAMBDA is not None, "Pas de λ choisi : voir la cellule 8"
!python cybervision_plausibility/entrainement.py --modele B --lambda {LAMBDA} --reprendre --sortie "{SORTIES}/modele_B_lambda{LAMBDA}"

## 10. Entraînement complet du modèle C (100 époques)
Modèle C = modèle B (même λ) + **critique à projection** (`critique_projection.py`) :
le critique note le réalisme **et** la cohérence avec la classe. Échantillonnage équilibré conservé.
En cas de déconnexion : réexécuter les cellules 1 à 3, puis celle-ci (`--reprendre`).

In [ ]:
LAMBDA_C = 0.1   # même λ que le modèle B (choisi à la cellule 8)
!python cybervision_plausibility/entrainement.py --modele C --lambda {LAMBDA_C} --reprendre --sortie "{SORTIES}/modele_C_lambda{LAMBDA_C}"